# PrintFarmEnv — SFT + GRPO Training Demo

**What this notebook does:**
1. Downloads the PrintFarmEnv codebase from HuggingFace
2. Runs a **mini SFT** warm-start (1 epoch, ~5 min on T4)
3. Runs a **mini GRPO** reinforcement loop (30 steps, ~15 min on T4)
4. Loads our **fully trained model** from HF Hub and demos inference

**Full production run** was done on HF Jobs (L4 GPU, 500 steps, ~6 hrs).  
Full artifacts: https://huggingface.co/sikkaBolega/printfarm-grpo-merged

> **Runtime:** Select `Runtime → Change runtime type → T4 GPU` before running.

In [ ]:
# Confirm GPU
import torch
assert torch.cuda.is_available(), "No GPU detected — change runtime to T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1 — Install dependencies

In [ ]:
%%capture
!pip install unsloth trl peft datasets huggingface_hub wandb -q

## 2 — Download PrintFarmEnv code from HuggingFace Space

In [ ]:
import os, sys
from huggingface_hub import snapshot_download

CODE_DIR = "/content/printfarm"

snapshot_download(
    repo_id="sikkaBolega/printfarm-env",
    repo_type="space",
    local_dir=CODE_DIR,
    ignore_patterns=["*.git*", "__pycache__"],
)

# Make the repo root importable
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.chdir(CODE_DIR)
print(f"Working directory: {os.getcwd()}")
print("Code downloaded.")

## 3 — SFT warm-start (mini run)

Trains a LoRA adapter on 125 balanced examples covering all 9 scorable actions.  
**~5 min on T4.** Full production run: 4 epochs on HF Jobs T4.

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"  # disable wandb for demo run

!python -m submission.training.train_sft_hf \
    --data submission/data/sft_warm.jsonl \
    --model Qwen/Qwen2.5-3B-Instruct \
    --epochs 1 \
    --batch_size 4 \
    --grad_accum 2 \
    --no_push \
    --out /content/sft_output

In [ ]:
# Verify SFT adapter saved
import os
adapter_files = os.listdir("/content/sft_output")
print("SFT output files:", adapter_files)

## 4 — GRPO reinforcement learning (mini run)

Runs 30 GRPO steps starting from our pre-merged SFT model (downloaded from HF Hub).  
Uses a 6-component composite reward: format, economic, fault precision, message handling, unnecessary action, novel fault.  
**~15 min on T4.** Full production run: 500 steps on HF Jobs L4.

Settings reduced for Colab T4 (16 GB VRAM):
- `n_generations=4` (vs 8 in production)
- `max_completion_length=128` (vs 256 in production)
- `max_steps=30` (vs 500 in production)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python -m submission.training.train_grpo_hf \
    --model Qwen/Qwen2.5-3B-Instruct \
    --init_model sikkaBolega/printfarm-sft-merged \
    --max_steps 30 \
    --n_prompts 50 \
    --n_generations 4 \
    --max_completion_length 128 \
    --max_seq_length 2048 \
    --temperature 0.9 \
    --save_steps 30 \
    --no_push \
    --out /content/grpo_output

In [ ]:
# Show reward trend from monitor log
import json, pathlib

monitor_path = pathlib.Path("/content/grpo_output/monitor.jsonl")
if monitor_path.exists():
    entries = [json.loads(l) for l in monitor_path.read_text().splitlines() if l]
    print(f"{'Step':>6}  {'reward_avg':>12}  {'parse':>8}  {'actions'}")
    print("-" * 60)
    for e in entries:
        print(f"{e.get('step', '?'):>6}  {e.get('reward_avg', 0):>+12.4f}  "
              f"{e.get('parse_pct', 0):>6.1f}%  {e.get('action_dist', {})}")
else:
    print("monitor.jsonl not found — check /content/grpo_output/")

## 5 — Inference with fully trained model

Loads the **production model** (500-step GRPO run on L4) from HF Hub.  
Demonstrates the dispatcher making decisions on 5 example scenarios.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER    = "sikkaBolega/printfarm-grpo-adapter"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Loading base model (4-bit)...")
# Strip bitsandbytes quantization_config if present (Unsloth artifact)
cfg = AutoConfig.from_pretrained(BASE_MODEL)
if getattr(cfg, "quantization_config", None) is not None:
    del cfg.quantization_config

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    config=cfg,
    torch_dtype=torch.float16,
    device_map="auto",
)

print(f"Loading GRPO adapter from {ADAPTER}...")
model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()
print("Ready.")

In [ ]:
from submission.shared.prompt import SYSTEM_PROMPT
from submission.shared.parse_action import parse_action

SCENARIOS = [
    # 1 — Fault detection
    ("Fault scenario",
     "STEP 15/60 | Net profit: $8.20\n\nPRINTERS:\n  - P1: IDLE, spool=1000g\n  - P2: PRINTING, mat=PLA, spool=700g, job=j2\n  - P3: IDLE, spool=1000g\n\nJOBS:\n  - j2: PRINTING, mat=PLA, progress=20%, $55, deadline_in=38steps\n\nOPERATOR NOTES:\n  - P2: grinding noise from extruder, unusual resistance"),

    # 2 — Rush order
    ("Rush order",
     "STEP 5/60 | Net profit: $1.50\n\nPRINTERS:\n  - P1: IDLE, spool=1000g\n  - P2: IDLE, spool=1000g\n\nJOBS:\n  - j1: PENDING, mat=PLA, weight=200g, time=15steps, $90, deadline_in=18steps, priority=HIGH\n\nCUSTOMER MESSAGES:\n  - [HIGH] job=j1: j1 — rush order, must ship today"),

    # 3 — Cancel request
    ("Cancel request",
     "STEP 10/60 | Net profit: $5.00\n\nPRINTERS:\n  - P1: PRINTING, mat=PETG, spool=800g, job=j3\n\nJOBS:\n  - j3: PRINTING, mat=PETG, progress=8%, $70, deadline_in=45steps\n  - j5: PENDING, mat=PLA, weight=180g, time=14steps, $40, deadline_in=20steps\n\nCUSTOMER MESSAGES:\n  - [?] job=j5: j5 — please cancel my order, changed my mind"),

    # 4 — High fatigue maintenance
    ("Maintenance needed",
     "STEP 30/90 | Net profit: $20.00\n\nPRINTERS:\n  - P1: IDLE, spool=900g, fatigue=8\n  - P2: PRINTING, mat=ABS, spool=600g, job=j4\n\nJOBS:\n  - j4: PRINTING, mat=ABS, progress=33%, $75, deadline_in=60steps"),

    # 5 — All healthy, queue manageable
    ("Wait (healthy state)",
     "STEP 25/60 | Net profit: $15.00\n\nPRINTERS:\n  - P1: PRINTING, mat=PLA, spool=800g, job=j1\n  - P2: PRINTING, mat=PETG, spool=650g, job=j2\n\nJOBS:\n  - j1: PRINTING, mat=PLA, progress=40%, $50, deadline_in=35steps\n  - j2: PRINTING, mat=PETG, progress=28%, $80, deadline_in=50steps, priority=HIGH"),
]

print(f"{'='*70}")
print(f"INFERENCE DEMO — Production model ({ADAPTER})")
print(f"{'='*70}\n")

for title, obs in SCENARIOS:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Current state:\n{obs}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=80, temperature=0.1,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    parsed = parse_action(completion)
    action = parsed.action_type if parsed else "PARSE_FAIL"

    print(f"[{title}]")
    print(f"  Output : {completion.strip()[:120]}")
    print(f"  Action : {action}")
    print()

## 6 — Quick evaluation (11 fixed seeds)

Runs the official eval harness on 11 episodes (same seeds used for submission scoring).

In [ ]:
# Uses the same eval script as the submission — loads adapter from HF Hub
!python -m submission.eval.eval_model \
    --model Qwen/Qwen2.5-3B-Instruct \
    --adapter sikkaBolega/printfarm-grpo-adapter \
    --n_episodes 11 \
    --output /content/eval_results.json

In [ ]:
import json
with open("/content/eval_results.json") as f:
    results = json.load(f)

print(f"Average reward : {results['avg_reward']:+.4f}")
print(f"Format failures: {results['format_failure_rate']:.1%}")
print(f"Action distribution: {results['action_distribution']}")
print(f"\nPer-episode breakdown:")
for ep in results["episodes"]:
    print(f"  Ep {ep['episode']:2d} | {ep['task_id']} | "
          f"{ep['parsed']['action_type'] if ep['parsed'] else 'FAIL':>22s} | "
          f"reward={ep['reward']:+.3f}")

## References

| Artifact | Link |
|---|---|
| HF Space (code + data) | https://huggingface.co/spaces/sikkaBolega/printfarm-env |
| SFT adapter | https://huggingface.co/sikkaBolega/printfarm-sft-adapter |
| SFT merged model | https://huggingface.co/sikkaBolega/printfarm-sft-merged |
| GRPO adapter (submission) | https://huggingface.co/sikkaBolega/printfarm-grpo-adapter |
| GRPO merged model (submission) | https://huggingface.co/sikkaBolega/printfarm-grpo-merged |